# Snowpark ML: Model Training and Grid Search

## Notebook 02 - Training and Evaluating a Model

Now we use Snowpark ML to train a classifier using the XGBoost estimator, evaluate its performance, and optimize hyperparameters with a grid search.

Steps:
1) Setup (preamble)
2) Train the Model
3) Evaluate the Model
4) Optimize Hyperparameters with Grid Search

## 1. Setup (preamble)

In [ ]:
# Snowpark for Python
from snowflake.snowpark import Session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark.types import StructType, StructField, DoubleType, StringType, DecimalType
from snowflake.snowpark.functions import *
from snowflake.snowpark.types import *

# Snowpark ML
import snowflake.ml.modeling.preprocessing as snowparkml
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.metrics.correlation import correlation
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.modeling.metrics import *

# General Data Science Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
import json
import joblib

# Warning Suppression
import warnings; warnings.simplefilter('ignore')

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

# current_user = session.get_current_user()
current_user = props['user']

## 2. Train the Model

- Create and use a Snowpark-optimized warehouse

For the heavier work of model training, let's create and use a Snowpark-optimized warehouse. In a typical account you may not be permitted to create a warehouse, but hopefully you can elect to use one created for you!

In [ ]:
# Define a warehouse name and create warehouse
MODELLING_WH = current_user + "_SP_OPTIMIZED_WH"
_ = session.sql("create or replace warehouse " + MODELLING_WH + " warehouse_type = 'snowpark-optimized'" +
                " warehouse_size = medium auto_suspend = 180 initially_suspended=True ").collect()

# Use the warehouse just created
_ = session.sql("use warehouse " + MODELLING_WH).collect()

* Define Input, Label and Output Columns

In [ ]:
# This is the training dataset table persisted from the previous notebook
trainDF = session.table(current_user + '_db.public.training_data')

# This is the test dataset table persisted from the previous notebook
testDF = session.table(current_user + '_db.public.test_data')

# Let's quickly look at the set of columns (features) in our training set
trainDF.columns

In [ ]:
FEATURE_COLUMNS = trainDF.drop('CHURNED').columns
LABEL_COLUMNS = ["CHURNED"]
OUTPUT_COLUMNS = ["PREDICTED_CHURN"]

- Train Snowpark ML XGBoost Classifier using fit

In [ ]:
%%time

classifier = XGBClassifier(
    input_cols = FEATURE_COLUMNS,
    label_cols = LABEL_COLUMNS,
    output_cols = OUTPUT_COLUMNS
)

# Train the model
classifier.fit(trainDF)


# The model can now be serialized and persisted; but we're not done yet... let's do some tests

We can see the underlying estimator by retrieving it with to_xgboost()

In [ ]:
classifier.to_xgboost()

- Generate predictions for test data using predict()

In [ ]:
result = classifier.predict(testDF)

In [ ]:
result.select('predicted_churn', 'churned').show()

## 3. Evaluate the Model

Use a few methods in Snowpark ML to evaluate the predictions made on the test set.

- Baseline **Confusion Matrix** (Columns: **Predictions** 0 and 1; Rows: **Actual** 0 and 1)

In [ ]:
confusion_data = result.select('churned', 'predicted_churn').to_pandas()

from sklearn.metrics import confusion_matrix
confusion_matrix(confusion_data['CHURNED'], confusion_data['PREDICTED_CHURN'])

- Baseline **Accuracy** (Fraction of all predictions that are correct)

In [ ]:
print('Accuracy:', accuracy_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- Baseline **Precision** (Fraction of true predictions that are correct)

In [ ]:
print('Precision:', precision_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- Baseline **Recall** (Fraction of actual true values that are predicted as true)

In [ ]:
print('Recall:', recall_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- Baseline **F1 score** (Harmonic mean of precision and recall, combining those evaluations into one number)

In [ ]:
print('F1 Score:', f1_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

An F1 score above 0.5 is of some value; above 0.8 is good; above 0.9 is excellent. 

## 4. Optimize Hyperparameters with a Grid Search

Let's see if we can find a better set of hyperparameters than what we trained with above.  Grid Search will perform an exhaustive search across all combinations of hyperparmeters we enumerate below, in parallel.

In [ ]:
%%time

from snowflake.ml.modeling.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=XGBClassifier(),

    # A total of 30 (6 * 5) models will be trained with combinations of hyperparameters specified below
    param_grid={
        "n_estimators":[10, 25, 50, 100, 150, 200], # Vary the n_estimators with each of these 6 n_estimators
        "max_depth":[1,3,5,7,9] # Vary the max_depth with each of these 5 max_depths
    },
    n_jobs = -1, # -1 means to run as many jobs in parallel as there are processors; the default is 1, or no parallelism
    scoring = "f1",
    input_cols = FEATURE_COLUMNS,
    label_cols = LABEL_COLUMNS,
    output_cols = OUTPUT_COLUMNS
)

# Do the grid search!
grid_search.fit(trainDF)

We can use to_sklearn() to unwrap the model from the grid_search

In [ ]:
grid_search.to_sklearn().best_estimator_

We can see the best estimator has found hyperparameters that produced a _better_ model. If we like, we can pursue further iterations with models other than XGBoost with this particular dataset, which the Snowpark ML framework makes easy.

Let's explore how the model performed as the n_estimators and max_depth were varied.

In [ ]:
# Analyze grid search results
gs_results = grid_search.to_sklearn().cv_results_
n_estimators_val = []
max_depth_val = []
for param_dict in gs_results["params"]:
    n_estimators_val.append(param_dict["n_estimators"])
    max_depth_val.append(param_dict["max_depth"])
f1_val = gs_results["mean_test_score"]

gs_results_df = pd.DataFrame(data={
    "n_estimators":n_estimators_val,
    "max_depth":max_depth_val,
    "f1":f1_val})

sns.relplot(data=gs_results_df, x="n_estimators", y="f1", hue="max_depth", kind="line")

plt.show()

## Evaluate the best estimator from the Grid Search using the same metrics

- Generate predictions for test data using predict()

In [ ]:
result = grid_search.predict(testDF)

In [ ]:
result.select('predicted_churn', 'churned').show()

## Evaluate the Best Model from the Grid Search

Compare to the metrics from the default model built prior to the grid search

- **Confusion matrix**

In [ ]:
confusion_data = result.select('churned', 'predicted_churn').to_pandas()

confusion_matrix(confusion_data['CHURNED'], confusion_data['PREDICTED_CHURN'])

- **Accuracy**

In [ ]:
print('Accuracy:', accuracy_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- **Precision**

In [ ]:
print('Precision:', precision_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- **Recall**

In [ ]:
print('Recall:', recall_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

- **F1 score**

In [ ]:
print('F1 Score:', f1_score(df=result, y_true_col_names='CHURNED', y_pred_col_names='PREDICTED_CHURN'))

If we're tuning for F1, the grid search has yielded some improvement in the model. 
At this point you can see how rapidly one could modify parameters and iterate with new approaches.

### Persist the Model

- Let's save our best model so far by serializing it (pickling) and writing out to a file.
- We will explore using the Model Registry in the next notebook, instead of writing models out to the local filesystem.

In [ ]:
joblib.dump(grid_search, 'grid_search.joblib')

In [ ]:
session.close()